# Optimización de un sistema RAG para consultas frecuentes y contexto largo

Una empresa tiene un sistema RAG interno que responde preguntas sobre documentación técnica de productos.
Actualmente, el sistema es funcional, pero enfrenta problemas:

* Algunas consultas frecuentes consumen demasiados recursos.

* Las respuestas a veces son largas y contienen información irrelevante.

* El costo de la API LLM es alto debido a inputs grandes.

El objetivo de la práctica es optimizar el flujo del generator y del retriever, aplicando técnicas de Prompt Engineering, compresión de contexto, caching y medición de métricas.

## Datos a utilizar

Para realizar la práctica, se usarán dos CSVs de ejemplo:

1. documentacion_auditoria.csv

Contiene documentos relacionados con auditorías, normativa y riesgos financieros.

Usar este dataset para consultas sobre auditoría, normativa, riesgos o control interno.


2. documentacion_tecnica.csv

Contiene documentos relacionados con sistemas RAG, arquitectura de software y optimización de procesos.

Usar este dataset para consultas sobre implementación técnica, arquitectura, procesamiento de datos o RAG multimodal.

## Objetivos de la práctica

1. Implementar Prompt Engineering avanzado:

- Diseñar prompts que obliguen al LLM a citar el contenido recuperado.

- Establecer roles, por ejemplo: “Eres un experto técnico que redacta informes precisos”.

In [ ]:
import os
import pandas as pd
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from pathlib import Path

LLM_MODEL = "gpt-3.5-turbo"
VECTOR_DB_PATH = "./chroma_db_rag1" 
CONTEXT_RETRIEVERS = 3

def load_and_prepare_data(file_auditoria: str, file_tecnica: str) -> list:
    """Carga los CSV de forma segura y combina el contenido."""
    try:
        df_auditoria = pd.read_csv(file_auditoria, encoding='utf-8', sep=',')
        df_tecnica = pd.read_csv(file_tecnica, encoding='utf-8', sep=',')
        
    except FileNotFoundError as e:
        print(f"❌ ERROR: No se encontró el archivo. Asegúrate de que las rutas sean correctas: {e}")
        exit(1)
    except pd.errors.ParserError:
        print("❌ ERROR: Problemas al parsear el CSV. Verifica que no haya comas sin comillas en el contenido.")
        exit(1)
        
    documents = []
    for index, row in df_auditoria.iterrows():
        documents.append(f"Documento Auditoría ID {row.get('id', 'N/A')} - Título: {row.get('titulo', 'N/A')}. Contenido: {row.get('contenido', 'N/A')}")
    for index, row in df_tecnica.iterrows():
        documents.append(f"Documento Técnico ID {row.get('id', 'N/A')} - Título: {row.get('titulo', 'N/A')}. Contenido: {row.get('contenido', 'N/A')}")
        
    return documents

def setup_vector_store(documents: list):
    """Divide el texto y configura la base de datos vectorial de forma robusta."""

    text_splitter = CharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=100,
        separator="\n",
        length_function=len
    )
    
    embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
    
    if Path(VECTOR_DB_PATH).exists() and len(list(Path(VECTOR_DB_PATH).iterdir())) > 0:
        print(f"✅ Cargando base de datos vectorial existente en {VECTOR_DB_PATH}...")
        vectorstore = Chroma(
            persist_directory=VECTOR_DB_PATH, 
            embedding_function=embeddings
        )
    else:
        print(f"🛠️ Creando nueva base de datos vectorial en {VECTOR_DB_PATH}...")
        chunks = [chunk for doc in documents for chunk in text_splitter.split_text(doc)]
        
        vectorstore = Chroma.from_texts(
            texts=chunks, 
            embedding=embeddings, 
            persist_directory=VECTOR_DB_PATH
        )
        vectorstore.persist()
        
    return vectorstore.as_retriever(search_kwargs={"k": CONTEXT_RETRIEVERS})


def objective_1_advanced_prompt_engineering(retriever):
    """Implementa Prompt Engineering con Rol y Citas."""
    print("\n--- 1. Prompt Engineering Avanzado (Con Citas) ---")
    
    SYSTEM_PROMPT = (
        "Eres un experto técnico que redacta informes precisos y concisos. "
        "Tu tarea es responder a la pregunta basándote **únicamente** en el contexto proporcionado. "
        "**DEBES** citar la fuente del documento utilizado (Título e ID) al final de cada frase o punto. "
        "Ejemplo de cita: (Documento Técnico ID 1, Manual de instalación)."
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "Contexto: {context}\n\nPregunta: {question}")
    ])
    
    if not os.getenv("OPENAI_API_KEY"):
         print("⚠️ WARNING: La variable de entorno 'OPENAI_API_KEY' no está configurada. El LLM fallará.")

    llm = ChatOpenAI(model=LLM_MODEL, temperature=0.1)
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff", 
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True
    )
    
    query = "¿Cuáles son los tipos de documentación de mantenimiento y cómo se instalan los sistemas?"
    print(f"Pregunta: {query}")
    
    try:
        response = qa_chain.invoke({"query": query})
        
        print("\n✅ Respuesta Generada (Debe contener citas):")
        print(response['result'])
        print("\nDocumentos Fuente:")
        for doc in response['source_documents']:
            print(f"- {doc.page_content[:100]}...")
    except Exception as e:
        print(f"❌ ERROR: Fallo al invocar al LLM (posiblemente clave API, límite de tokens o red): {e}")


# --- Ejecución ---
if __name__ == "__main__":
    FILE_AUDITORIA = './Data/documentacion_auditoria.csv'
    FILE_TECNICA = './Data/documentacion_tecnica.csv'
    
    print("Iniciando carga de datos y configuración RAG...")
    documents = load_and_prepare_data(FILE_AUDITORIA, FILE_TECNICA)
    retriever = setup_vector_store(documents)
    
    objective_1_advanced_prompt_engineering(retriever)
    
    print("\nProceso de Solución 1 finalizado.")

2. Aplicar compresión de contexto:

- Reducir la cantidad de tokens enviados al LLM extrayendo solo la información relevante de los documentos recuperados.

- Comparar eficiencia y costo antes y después de la compresión.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
import time

def objective_2_context_compression(base_retriever):
    """
    Implementa compresión de contexto para reducir el número de tokens enviados al LLM.
    Utiliza un LLMChainExtractor para condensar la información.
    """
    print("\n--- 2. Aplicar Compresión de Contexto ---")

    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0) 
    compressor = LLMChainExtractor.from_llm(llm)
    

    base_retriever.search_kwargs={"k": 10} 
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, 
        base_retriever=base_retriever
    )
    
    query = "Dime qué documentos tratan sobre el reinicio de sistemas y su versión."
    
    docs_base = base_retriever.get_relevant_documents(query)
    tokens_base = sum(len(doc.page_content.split()) for doc in docs_base) # Estimación
    print(f"1. Sin Compresión: Recuperados {len(docs_base)} chunks. Tokens estimados: {tokens_base}")
    
    start_time = time.time()
    docs_compressed = compression_retriever.get_relevant_documents(query)
    end_time = time.time()
    tokens_compressed = sum(len(doc.page_content.split()) for doc in docs_compressed) # Estimación
    
    print(f"2. Con Compresión: Recuperados {len(docs_compressed)} chunks. Tokens estimados: {tokens_compressed}")
    print(f"Latencia de Compresión: {end_time - start_time:.2f} segundos.")
    
    print(f"\n✅ Reducción de Tokens: Se pasa de ≈{tokens_base} tokens a ≈{tokens_compressed} tokens (ahorro significativo en el prompt final).")
    print("Contenido Comprimido (Primer chunk):")
    print(docs_compressed[0].page_content)


# --- Ejecución ---
if __name__ == "__main__":
    documents = load_and_prepare_data('./Data/documentacion_auditoria.csv', './Data/documentacion_tecnica.csv')
    base_retriever = setup_vector_store(documents)
    objective_2_context_compression(base_retriever)

3. Implementar caching de respuestas:

- Guardar respuestas a consultas frecuentes y reutilizarlas sin invocar al LLM nuevamente.

- Medir reducción de latencia y uso de recursos.

In [ ]:

from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.globals import set_llm_cache
from langchain_community.cache import InMemoryCache 
import time


def objective_3_implement_caching(retriever):
    """Implementa caching de respuestas para reducir latencia y uso de recursos."""
    print("\n--- 3. Implementación de Caching ---")
    
    set_llm_cache(InMemoryCache())
    print("Caching de LLM habilitado (InMemoryCache).")
    
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    
    query = "Dime los pasos para la configuración de red y las políticas de actualización."

    print("\nPrimera consulta (sin cache)...")
    start_time_1 = time.time()
    qa_chain.invoke({"query": query}) 
    end_time_1 = time.time()
    latencia_1 = end_time_1 - start_time_1
    print(f"Latencia (1ª vez): {latencia_1:.2f} segundos")
    
    print("\nSegunda consulta (con cache)...")
    start_time_2 = time.time()
    qa_chain.invoke({"query": query})
    end_time_2 = time.time()
    latencia_2 = end_time_2 - start_time_2
    print(f"Latencia (2ª vez - Cacheada): {latencia_2:.2f} segundos")
    
    print(f"\n✅ Reducción de Latencia: {(latencia_1 - latencia_2):.2f} segundos. Esto demuestra el ahorro de recursos del LLM.")

# --- Ejecución ---
if __name__ == "__main__":
    documents = load_and_prepare_data('./Data/documentacion_auditoria.csv', './Data/documentacion_tecnica.csv')
    retriever = setup_vector_store(documents)
    objective_3_implement_caching(retriever)

4. Medir métricas de calidad:

- Fidelidad (Faithfulness): verificar que la respuesta esté basada en los documentos recuperados.

- Contexto relevante: asegurar que solo se use información necesaria.

- Respuesta relevante: verificar que la consulta esté contestada correctamente.

In [ ]:
import os
from ragas.langchain.evalchain import RagasEvaluatorChain
from ragas.metrics import (
    faithfulness,         # Fidelidad (Faithfulness)
    answer_relevancy,     # Respuesta relevante (Answer Relevance)
    context_relevancy,    # Contexto relevante (Context Relevancy)
)
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from rag_solution_base import load_and_prepare_data, setup_vector_store 

EVALUATOR_LLM = ChatOpenAI(model="gpt-4o-mini", temperature=0) 

faithfulness_chain = RagasEvaluatorChain(metric=faithfulness, llm=EVALUATOR_LLM)
answer_relevancy_chain = RagasEvaluatorChain(metric=answer_relevancy, llm=EVALUATOR_LLM)
context_relevancy_chain = RagasEvaluatorChain(metric=context_relevancy, llm=EVALUATOR_LLM)

def objective_4_measure_quality(retriever):
    """
    Mide métricas de calidad (Faithfulness, Context Relevancy, Answer Relevancy) 
    utilizando RAGAS a través de LangChain.
    """
    print("\n--- 4. Medición de Métricas de Calidad (RAGAS) ---")
    
    if not os.getenv("OPENAI_API_KEY"):
         print("⚠️ WARNING: La variable 'OPENAI_API_KEY' no está configurada. El LLM fallará.")
         return

    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm, 
        chain_type="stuff", 
        retriever=retriever,
        return_source_documents=True
    )

    query = "Dime qué documentos tratan sobre el reinicio de sistemas, la seguridad eléctrica y la configuración de red."
    print(f"Pregunta a Evaluar: {query}")
    
    try:
        rag_response = qa_chain.invoke({"query": query})
    except Exception as e:
        print(f"❌ ERROR: Fallo al invocar al LLM (posiblemente clave API o red): {e}")
        return

    print("\nCalculando métricas con RAGAS (puede tomar un momento)...")
    
    faithfulness_score = faithfulness_chain.evaluate(rag_response)
    print(f"\n- Fidelidad (Faithfulness): {faithfulness_score['faithfulness_score']:.4f}")

    context_relevancy_score = context_relevancy_chain.evaluate(rag_response)
    print(f"- Contexto Relevante (Context Relevancy): {context_relevancy_score['context_relevancy_score']:.4f}")
    
    answer_relevancy_score = answer_relevancy_chain.evaluate(rag_response)
    print(f"- Respuesta Relevante (Answer Relevancy): {answer_relevancy_score['answer_relevancy_score']:.4f}")
    
    print("\n--- Resultado de la Respuesta RAG ---")
    print(rag_response['result'])
    print("\n--- Interpretación de los Scores ---")
    print("Scores cercanos a 1.0 indican alta calidad en la métrica respectiva.")

# --- Ejecución ---
if __name__ == "__main__":
    FILE_AUDITORIA = './Data/documentacion_auditoria.csv'
    FILE_TECNICA = './Data/documentacion_tecnica.csv'
  
    try:
        documents = load_and_prepare_data(FILE_AUDITORIA, FILE_TECNICA)
        retriever = setup_vector_store(documents)
        objective_4_measure_quality(retriever)
    except NameError:
        print("\n❌ ERROR: Asegúrate de tener instalada la librería 'ragas' (`pip install ragas`) ")

5. Experimentación iterativa (MLOps):

- Probar distintos modelos de embeddings y LLM para evaluar trade-offs entre calidad y costo.

- Ajustar número de documentos recuperados, re-ranking y tamaño de chunks.

- Comparar dos versiones del sistema (A/B Testing) y reportar mejoras.

In [ ]:
import time
import tiktoken
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA


def run_test(version_name: str, llm_model: str, k_docs: int, retriever):
    """Ejecuta una prueba RAG con parámetros específicos y mide métricas."""
    print(f"\n--- Ejecutando Prueba: {version_name} ---")
    
    retriever.search_kwargs={"k": k_docs}
    llm = ChatOpenAI(model=llm_model, temperature=0.1)
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    
    query = "¿Qué procedimientos de mantenimiento existen y en qué documento se encuentran?"
    
    start_time = time.time()
    response = qa_chain.invoke({"query": query})
    end_time = time.time()
    latencia = end_time - start_time
    

    input_context_text = " ".join([doc.page_content for doc in retriever.get_relevant_documents(query)])
    encoding = tiktoken.encoding_for_model(llm_model)
    
    input_tokens = len(encoding.encode(input_context_text + query))
    output_tokens = len(encoding.encode(response['result']))
    
    print(f"Modelo LLM: {llm_model} | K Docs: {k_docs}")
    print(f"Latencia Total: {latencia:.2f} segundos")
    print(f"Tokens de Entrada (Costo): {input_tokens}")
    print(f"Tokens de Salida (Costo): {output_tokens}")
    
    return latencia, input_tokens, output_tokens

def objective_5_ab_testing(retriever):
    """Compara dos versiones del sistema (A y B)."""
    
    lat_A, in_A, out_A = run_test("A: Base (gpt-3.5, k=3)", "gpt-3.5-turbo", 3, retriever)
    lat_B, in_B, out_B = run_test("B: Calidad (gpt-4, k=5)", "gpt-4o", 5, retriever)
    
    print("\n--- Reporte Final de A/B Testing ---")
    if lat_A > lat_B:
        print(f"✅ La **Versión B** es {(lat_A - lat_B)/lat_A * 100:.1f}% más rápida que A.")
    else:
        print(f"❌ La **Versión A** es más rápida, pero la Versión B tiene un costo de tokens {(in_B - in_A)/in_A * 100:.1f}% mayor.")
    

# --- Ejecución ---
if __name__ == "__main__":
    # Necesitas ejecutar la configuración base
    documents = load_and_prepare_data('./Data/documentacion_auditoria.csv', './Data/documentacion_tecnica.csv')
    retriever = setup_vector_store(documents)
    objective_5_ab_testing(retriever)

6. Opcional avanzado – RAG Multimodal:

- Indexar documentos con imágenes/diagramas y permitir consultas que incluyan referencias visuales.

- Generar respuestas que incluyan la descripción del diagrama relevante.